# Fine-tune Qwen2.5-1.5B-Instruct (LoRA) trên UIT-ViQuAD2.0

Notebook end-to-end cho **Extractive QA** (UIT-ViQuAD2.0 / SQuAD 2.0) trên [taidng/UIT-ViQuAD2.0](https://huggingface.co/datasets/taidng/UIT-ViQuAD2.0).

**Pipeline:** HF dataset → token profiling → **LoRA fp16/bf16 (không quantize)** → lưu adapter → EM/F1 eval.

**Câu không có đáp án** (`is_impossible=true`): `"Không có đáp án trong đoạn văn"`.

### Cách chạy trên máy thuê GPU
1. Chạy **Bước 1 → 2 → 3** (pip install)
2. **Kernel → Restart Kernel**
3. Chạy **Bước 3b** (verify) rồi **Run All** phần còn lại
4. OOM: `per_device_train_batch_size=1`, `gradient_accumulation_steps=8`, hoặc `MAX_SEQ_CAP=3584`

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
# Bước 2: PyTorch CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# Bước 3: transformers, trl, unsloth, ...
!pip install transformers trl peft accelerate bitsandbytes xformers datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn --no-cache-dir

In [ ]:
# Bước 3b: Verify sau pip — nếu FAIL → Restart Kernel rồi chạy lại cell này
import importlib

for pkg in ["torch", "transformers", "datasets", "unsloth"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

In [ ]:
# Tắt warnings & log rác — chạy cell này trước mọi cell khác (sau pip + restart kernel)
import warnings
import logging
import os
import sys

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

logging.captureWarnings(True)
logging.getLogger("py.warnings").setLevel(logging.ERROR)

for _name in (
    "transformers", "transformers.modeling_utils", "transformers.tokenization_utils_base",
    "transformers.generation.utils", "datasets", "urllib3", "torch", "triton",
    "unsloth", "peft", "accelerate", "huggingface_hub", "httpx", "filelock",
):
    logging.getLogger(_name).setLevel(logging.ERROR)

try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

if not sys.warnoptions:
    warnings.simplefilter("ignore")

print("Warnings suppressed.")

In [ ]:
import json
import math
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ========== CẤU HÌNH ==========
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

SYSTEM_PROMPT = (
    "Bạn là một trợ lý AI thông minh, giỏi tiếng Việt. "
    "Nhiệm vụ của bạn là đọc đoạn văn và trả lời câu hỏi "
    "bằng cách trích xuất chính xác đoạn văn bản ngắn nhất từ ngữ cảnh được cung cấp. "
    f"Nếu câu hỏi không thể trả lời từ đoạn văn, hãy trả lời: {NO_ANSWER_SENTINEL}."
)

USER_PROMPT_TEMPLATE = (
    "Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "
    "đúng cụm từ xuất hiện trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn "
    f"hoặc '{NO_ANSWER_SENTINEL}' nếu không có đáp án:"
)

LORA_SAVE_PATH = "qwen2.5-1.5b-instruct-lora-viquad2"
EVAL_RESULTS_PATH = "eval_results_viquad2.json"
PROFILING_CONFIG_PATH = "profiling_config.json"

DATASET_ROOT = Path("../../")
DEV_JSON_PATH = DATASET_ROOT / "dev_viquad2.json"
TEST_JSON_PATH = DATASET_ROOT / "test_viquad2.json"

RUN_TRAINING = True
RUN_EVALUATION = True

# Khi train lại: ép tải HF + export lại JSON (tránh dùng file cũ sai nhãn)
FORCE_HF_DATASET = True
FORCE_REEXPORT_JSON = True

LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512

# Thanh tiến trình thống nhất cho profiling & evaluation
TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"

def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def sample_to_train_text(sample, tok):
    messages = build_messages(sample["context"], sample["question"], sample["answer"])
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def load_tokenizer(model_path=BASE_MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} thiếu chat_template.")
    return tok

In [ ]:
# ========== TẢI UIT-ViQuAD2.0 TỪ HUGGINGFACE ==========
# Fix quan trọng: nếu `answers.text` rỗng => phải coi là `is_impossible=True`

def row_to_sample(row):
    is_impossible = bool(row["is_impossible"])

    answers_obj = row.get("answers") or {}
    texts = answers_obj.get("text") or []

    if is_impossible or len(texts) == 0:
        # Unanswerable (SQuAD2): không có đáp án trong context
        is_impossible = True
        answer = NO_ANSWER_SENTINEL
    else:
        answer = texts[0]
        # phòng trường hợp answer bị rỗng/space
        if not str(answer).strip():
            is_impossible = True
            answer = NO_ANSWER_SENTINEL

    return {
        "id": row.get("id", ""),
        "title": row.get("title", ""),
        "context": row["context"],
        "question": row["question"],
        "answer": answer,
        "is_impossible": is_impossible,
    }


def hf_split_to_samples(split_dataset):
    return [row_to_sample(row) for row in split_dataset]


def save_samples_json(samples, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Đã lưu {len(samples)} mẫu → {path.resolve()}")


def split_stats(name, samples):
    n = len(samples)
    n_no = sum(1 for s in samples if s["is_impossible"])
    print(f"{name}: {n} | NoAns: {n_no} ({100*n_no/max(n,1):.1f}%)")


try:
    print(f"Đang tải {DATASET_NAME} ...")
    raw_ds = load_dataset(DATASET_NAME)
    train_samples = hf_split_to_samples(raw_ds["train"])
    dev_samples = hf_split_to_samples(raw_ds["validation"])
    test_samples = hf_split_to_samples(raw_ds["test"])
except Exception as e:
    if FORCE_HF_DATASET:
        raise RuntimeError(f"FORCE_HF_DATASET=True nhưng không tải được HF: {e}")

    print(f"HF lỗi ({e}). Thử JSON local...")
    if not TEST_JSON_PATH.exists():
        raise
    with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
        test_samples = json.load(f)
    with open(DEV_JSON_PATH, "r", encoding="utf-8") as f:
        dev_samples = json.load(f)
    train_samples = dev_samples
    print("⚠ Đang dùng JSON local (có thể sai nhãn) — khuyến nghị bật FORCE_HF_DATASET=True khi train lại.")

split_stats("Train", train_samples)
split_stats("Val", dev_samples)
split_stats("Test", test_samples)

# Luôn export lại JSON khi train lại để tránh file cũ
if FORCE_REEXPORT_JSON:
    save_samples_json(dev_samples, DEV_JSON_PATH)
    save_samples_json(test_samples, TEST_JSON_PATH)
    print("✅ Re-export JSON done.")
else:
    print("SKIP export JSON (FORCE_REEXPORT_JSON=False)")

# Sanity: Test phải có NoAns, nếu 0 thì mapping/export đang sai
n_no_test = sum(1 for s in test_samples if s["is_impossible"])
if n_no_test == 0:
    print("⚠ Test split có 0 NoAns. Điều này thường là do bạn đang dùng JSON cũ sai nhãn. Hãy dùng HF dataset + re-export.")

In [ ]:
# ========== PROFILING TOKEN LENGTH (chỉ transformers, không unsloth) ==========
# Lưu ý: phải chạy cell tải dataset trước để có `train_samples`.

if "train_samples" not in globals():
    raise NameError(
        "Chưa có biến `train_samples`. Hãy chạy cell 'TẢI UIT-ViQuAD2.0 TỪ HUGGINGFACE' trước (cell ngay phía trên), rồi chạy lại profiling."
    )


def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    total = len(samples)
    pbar = tqdm(samples, total=total, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    for i, s in enumerate(pbar, start=1):
        text = sample_to_train_text(s, tok)
        lengths.append(len(tok.encode(text)))
        pbar.set_postfix(done=f"{i}/{total}")
    lengths.sort()
    n = len(lengths)
    stats = {
        "min": lengths[0],
        "p50": lengths[n // 2],
        "p95": lengths[int(n * 0.95)],
        "p99": lengths[int(n * 0.99)],
        "max": lengths[-1],
    }
    chosen = min(math.ceil(stats["p99"] * 1.05), cap)
    chosen = ((chosen + 255) // 256) * 256
    chosen = max(chosen, min_len)
    truncated = sum(1 for L in lengths if L > chosen)
    stats["chosen_max_seq_length"] = chosen
    stats["truncated_samples"] = truncated
    stats["truncated_pct"] = round(100 * truncated / n, 3)
    return chosen, stats


tokenizer_prof = load_tokenizer()

if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    with open(PROFILING_CONFIG_PATH, "r", encoding="utf-8") as f:
        prof_cfg = json.load(f)
    max_seq_length = prof_cfg["max_seq_length"]
    length_stats = prof_cfg["token_length_stats"]
    print(f"Load profiling: max_seq_length={max_seq_length}")
else:
    max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
    with open(PROFILING_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats}, f, indent=2)

print("\n--- Token stats ---")
for k, v in length_stats.items():
    print(f"  {k}: {v}")
print(f"\n=> max_seq_length = {max_seq_length}")

In [ ]:
# ========== LOAD MODEL + LoRA (Unsloth, không quantize) ==========
if RUN_TRAINING:
    import warnings
    warnings.filterwarnings("ignore")

    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

    print("Model + LoRA loaded | quantize:", LOAD_IN_4BIT or LOAD_IN_8BIT)
    print("max_seq_length:", max_seq_length)
else:
    print("RUN_TRAINING=False — skip model load.")

In [ ]:
# ========== CHUẨN BỊ DATASET SFT ==========
if RUN_TRAINING:
    from datasets import Dataset

    def formatting_prompts_func(examples):
        texts = []
        for ctx, q, ans in zip(examples["context"], examples["question"], examples["answer"]):
            msgs = build_messages(ctx, q, ans)
            texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
        return {"text": texts}

    train_hf = Dataset.from_list(train_samples)
    dataset = train_hf.map(
        formatting_prompts_func,
        batched=True,
        remove_columns=train_hf.column_names,
        desc="Format prompts",
    )
    print(f"Train samples: {len(dataset)}")
    print(dataset[0]["text"][:500])

In [ ]:
# ========== HUẤN LUYỆN ==========
if RUN_TRAINING:
    import warnings
    warnings.filterwarnings("ignore")

    from trl import SFTTrainer
    from transformers import TrainingArguments
    from unsloth import is_bfloat16_supported
    import sys

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=1,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            num_train_epochs=3,
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=20,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=3407,
            output_dir="outputs_viquad2",
            save_strategy="epoch",
            report_to="none",
        ),
    )

    cfg_cls = type(trainer.args)
    tr_cls = type(trainer)
    sys.modules["trl.trainer.sft_config"] = sys.modules[cfg_cls.__module__]
    sys.modules["trl.trainer.sft_trainer"] = sys.modules[tr_cls.__module__]
    sys.modules[cfg_cls.__module__].SFTConfig = cfg_cls
    sys.modules[tr_cls.__module__].SFTTrainer = tr_cls

    print("Bắt đầu training...")
    trainer.train()
    print("Training xong.")

In [ ]:
# ========== SMOKE TEST + LƯU ADAPTER ==========
if RUN_TRAINING:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

    def run_inference(ctx, q, tok, mdl):
        msgs = build_messages(ctx, q, answer=None, for_inference=True)
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(mdl.device)
        mdl.generation_config.max_length = None
        mdl.generation_config.max_new_tokens = 64
        with torch.no_grad():
            out = mdl.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                max_new_tokens=64,
                do_sample=False,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )
        pred = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        return pred.split("\n")[0].strip()

    s1 = next(s for s in train_samples if not s["is_impossible"])
    s2 = next(s for s in train_samples if s["is_impossible"])
    print("Answerable pred:", run_inference(s1["context"], s1["question"], tokenizer, model))
    print("Truth:", s1["answer"])
    print("NoAns pred:", run_inference(s2["context"], s2["question"], tokenizer, model))
    print("Truth:", s2["answer"])

    model.save_pretrained(LORA_SAVE_PATH)
    tokenizer.save_pretrained(LORA_SAVE_PATH)
    print(f"Saved adapter → {LORA_SAVE_PATH}")

## Evaluation (UIT-ViQuAD2.0 test)
- **HasAns EM/F1** — câu có đáp án
- **NoAns accuracy** — câu `is_impossible=true`

Eval-only: `RUN_TRAINING=False`, `RUN_EVALUATION=True`

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def compute_em(pred, truth):
    return int(normalize_text(pred) == normalize_text(truth))


def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if n == 0: return 0.0
    p, r = n / len(pt), n / len(tt)
    return 2 * p * r / (p + r)


def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)

In [ ]:
if RUN_EVALUATION:
    import warnings
    warnings.filterwarnings("ignore")

    from transformers import AutoModelForCausalLM
    from peft import PeftModel

    if RUN_TRAINING:
        from unsloth import FastLanguageModel
        model_eval, tokenizer_eval = model, tokenizer
        FastLanguageModel.for_inference(model_eval)
    else:
        tok_path = LORA_SAVE_PATH if Path(LORA_SAVE_PATH).exists() else BASE_MODEL_NAME
        tokenizer_eval = load_tokenizer(tok_path)
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
        model_eval = PeftModel.from_pretrained(base, LORA_SAVE_PATH)
        model_eval.eval()

    # Model mặc định có generation_config.max_length=32768 → conflict với max_new_tokens → spam warning mỗi sample
    gen_cfg = model_eval.generation_config
    gen_cfg.max_length = None
    gen_cfg.max_new_tokens = 64

    def generate_prediction(inputs):
        return model_eval.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer_eval.pad_token_id,
            eos_token_id=tokenizer_eval.eos_token_id,
        )

    has_em, has_f1, no_ok, preds = [], [], [], []
    total_eval = len(test_samples)
    print(f"Bắt đầu evaluation: {total_eval} câu test\n")

    pbar = tqdm(
        test_samples,
        total=total_eval,
        desc="Evaluating",
        unit="sample",
        bar_format=TQDM_BAR,
    )

    for i, s in enumerate(pbar, start=1):
        pbar.set_description(f"Evaluating {i}/{total_eval}")

        msgs = build_messages(s["context"], s["question"], for_inference=True)
        prompt = tokenizer_eval.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_eval(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model_eval.device)
        with torch.no_grad():
            out = generate_prediction(inputs)
        pred = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0].strip()

        if s["is_impossible"]:
            ok = int(is_no_answer(pred))
            no_ok.append(ok); em, f1 = ok, float(ok)
        else:
            em = compute_em(pred, s["answer"]); f1 = compute_f1(pred, s["answer"])
            has_em.append(em); has_f1.append(f1)

        preds.append({"id": s["id"], "question": s["question"], "is_impossible": s["is_impossible"],
                      "ground_truth": s["answer"], "prediction": pred, "em": em, "f1": f1})

        postfix = {}
        if has_em:
            postfix["has_em"] = f"{100 * sum(has_em) / len(has_em):.1f}%"
        if no_ok:
            postfix["noans"] = f"{100 * sum(no_ok) / len(no_ok):.1f}%"
        pbar.set_postfix(postfix)

    n_has = len(has_em)
    n_no = len(no_ok)

    metrics = {
        "hasans_em": round(100 * sum(has_em) / max(n_has, 1), 4),
        "hasans_f1": round(100 * sum(has_f1) / max(n_has, 1), 4),
        # nếu test không có NoAns thì ghi None thay vì 0
        "noans_accuracy": (round(100 * sum(no_ok) / n_no, 4) if n_no > 0 else None),
        "overall_em": round(100 * sum(p["em"] for p in preds) / len(preds), 4),
        "overall_f1": round(100 * sum(p["f1"] for p in preds) / len(preds), 4),
        "n_hasans": n_has,
        "n_noans": n_no,
    }

    # ===== Pretty report (giống format bạn muốn) =====
    has_f1_buckets = {
        "0-0.2": 0,
        "0.2-0.5": 0,
        "0.5-0.8": 0,
        "0.8-1.0": 0,
        "1.0": 0,
    }
    for f1 in has_f1:
        if f1 == 1.0:
            has_f1_buckets["1.0"] += 1
        elif f1 >= 0.8:
            has_f1_buckets["0.8-1.0"] += 1
        elif f1 >= 0.5:
            has_f1_buckets["0.5-0.8"] += 1
        elif f1 >= 0.2:
            has_f1_buckets["0.2-0.5"] += 1
        else:
            has_f1_buckets["0-0.2"] += 1

    line = "=" * 50
    print("\n" + line)
    print("   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - UIT-ViQuAD2.0")
    print("   Model: Qwen2.5-1.5B-Instruct + LoRA (Unsloth)")
    print(line)
    print(f"Tổng số câu hỏi test : {total_eval}")
    print(f"HasAns (có đáp án)   : {n_has}")
    print(f"NoAns  (không đáp án): {n_no}")
    print(f"Exact Match (HasAns) : {metrics['hasans_em']:.2f}%")
    print(f"F1-score (HasAns)    : {metrics['hasans_f1']:.2f}%")
    if metrics["noans_accuracy"] is None:
        print("NoAns accuracy       : N/A (test split không có NoAns)")
    else:
        print(f"NoAns accuracy        : {metrics['noans_accuracy']:.2f}%")
    print(f"Overall EM            : {metrics['overall_em']:.2f}%")
    print(f"Overall F1            : {metrics['overall_f1']:.2f}%")
    print(f"max_seq_length        : {max_seq_length}")
    print(line)

    if n_has > 0:
        em_exact = sum(has_em)
        print(f"\nEM đúng hoàn toàn (HasAns): {em_exact}/{n_has} câu")
        print("\nPhân phối F1-score (HasAns):")
        for bucket, count in has_f1_buckets.items():
            pct = (count / n_has * 100) if n_has else 0
            print(f"F1 = {bucket}: {count} câu ({pct:.1f}%)")

    # Save JSON
    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump({"dataset": DATASET_NAME, "adapter": LORA_SAVE_PATH, "max_seq_length": max_seq_length,
                     "token_length_stats": length_stats, **metrics, "total": len(preds), "predictions": preds},
                    f, ensure_ascii=False, indent=2)
    print(f"\nKết quả chi tiết đã được lưu tại: {EVAL_RESULTS_PATH}")